# Module 08: Results, Diagnostics, and Publication Figures

## Learning to Autolens

---

**Purpose:** Extract science from your lens model fits. Learn to produce
corner plots, residual diagnostics, Einstein mass measurements, and
publication-quality figures that communicate your results clearly.

**Prerequisites:**
- Module 03 (result objects, posteriors)
- Module 04 (SLaM pipeline results)

**Key references:**
- Nightingale+18 Sec. 7: *Results and diagnostics*
- Foreman-Mackey (2016): *corner.py* — corner plot library

**Companion LaTeX notes:** `../../Notes/08_Results/08_results_theory.tex`

---

## Table of Contents

1. [Anatomy of a Result Object](#1-result-object)
2. [Corner Plots: Posterior Distributions](#2-corner-plots)
3. [Residual Analysis: Is the Model Good?](#3-residual-analysis)
4. [Physical Quantities: Einstein Mass and Radius](#4-physical-quantities)
5. [Source Plane Reconstruction](#5-source-plane)
6. [Publication-Quality Figures](#6-publication-figures)
7. [Model Comparison: Bayesian Evidence](#7-model-comparison)
8. [Exporting Results to CSV and FITS](#8-exporting)
9. [Exercises](#9-exercises)

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import autolens as al
import autolens.plot as aplt
import autofit as af

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

print(f"PyAutoLens version: {al.__version__}")

PyAutoLens version: 2025.11.18.1


In [2]:
# ============================================================
# LOAD DATA AND RUN A QUICK FIT (or load saved results)
# ============================================================
# For this module, we need a completed fit result.
# Either run a quick fit here or load results from Module 03/04.
# ============================================================

dataset_path = Path("../../autolens_workspace_original/dataset/imaging/simple__no_lens_light")

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=3.0,
)
dataset = dataset.apply_mask(mask=mask)

# Quick fit for demonstration
model = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(al.Galaxy, redshift=0.5,
                      mass=al.mp.Isothermal, shear=al.mp.ExternalShear),
        source=af.Model(al.Galaxy, redshift=1.0, bulge=al.lp.SersicCore),
    )
)

search = af.Nautilus(
    path_prefix=Path("output") / "module_08",
    name="results_demo",
    n_live=100,
)

result = search.fit(model=model, analysis=al.AnalysisImaging(dataset=dataset))
print("Fit complete!")

FileNotFoundError: [Errno 2] No such file or directory: '../../autolens_workspace_original/dataset/imaging/simple__no_lens_light/data.fits'

---

## 1. Anatomy of a Result Object <a id="1-result-object"></a>

The `result` object returned by `search.fit()` contains everything:

| Attribute | Type | What it gives you |
|-----------|------|-------------------|
| `result.info` | str | Text summary of all parameters |
| `result.max_log_likelihood_instance` | object | Best-fit parameter values |
| `result.max_log_likelihood_fit` | FitImaging | Best-fit model vs. data |
| `result.max_log_likelihood_tracer` | Tracer | Best-fit lens model |
| `result.samples` | Samples | Full posterior samples |
| `result.samples.log_evidence` | float | Bayesian evidence ln(Z) |

In [3]:
# ============================================================
# RESULT INFO: PARAMETER SUMMARY
# ============================================================
print(result.info)

NameError: name 'result' is not defined

---

## 2. Corner Plots: Posterior Distributions <a id="2-corner-plots"></a>

### Reading a Corner Plot

The corner plot shows:
- **Diagonal**: 1D marginal posterior for each parameter (histogram)
- **Off-diagonal**: 2D joint posteriors (contour plots)
- **Elongated contours**: parameter degeneracies
- **Circular contours**: independent parameters

In [4]:
# ============================================================
# CORNER PLOT
# ============================================================
# The NestPlotter produces corner plots from the posterior
# samples. Look for:
#   - Well-peaked 1D distributions → well-constrained
#   - Banana-shaped 2D → degeneracy (e.g., ε vs γ)
#   - Railing against edges → prior too narrow
# ============================================================

plotter = aplt.NestPlotter(samples=result.samples)
plotter.corner_anesthetic()

NameError: name 'result' is not defined

---

## 3. Residual Analysis: Is the Model Good? <a id="3-residual-analysis"></a>

### Diagnostic Checklist

1. **Normalized residuals**: should be $\mathcal{N}(0, 1)$ — Gaussian with σ=1
2. **No spatial structure**: residuals should look like random noise
3. **$\chi^2_{\rm red} \approx 1$**: neither over- nor under-fitting
4. **Residual histogram**: should be Gaussian

In [5]:
# ============================================================
# FIT SUBPLOT: THE STANDARD DIAGNOSTIC FIGURE
# ============================================================
fit_plotter = aplt.FitImagingPlotter(fit=result.max_log_likelihood_fit)
fit_plotter.subplot_fit()

NameError: name 'result' is not defined

In [6]:
# ============================================================
# RESIDUAL HISTOGRAM
# ============================================================
# The normalized residuals should follow N(0,1).
# Deviations indicate model mismatch.
# ============================================================

fit = result.max_log_likelihood_fit
norm_residuals = fit.normalized_residual_map.slim

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(norm_residuals, bins=50, density=True, alpha=0.7, color='steelblue',
        label="Normalized residuals")

# Overlay expected Gaussian
x = np.linspace(-5, 5, 200)
ax.plot(x, np.exp(-x**2/2) / np.sqrt(2*np.pi), 'r-', lw=2,
        label=r"$\mathcal{N}(0, 1)$")

ax.set_xlabel("Normalized residual $(d_i - m_i) / \sigma_i$", fontsize=12)
ax.set_ylabel("Probability density", fontsize=12)
ax.set_title("Residual Distribution", fontsize=13)
ax.legend(fontsize=11)

chi2 = fit.chi_squared
n_pix = fit.mask.pixels_in_mask
n_params = model.total_free_parameters
chi2_red = chi2 / (n_pix - n_params)

ax.text(0.02, 0.95, f"$\\chi^2_{{red}}$ = {chi2_red:.3f}\n"
        f"mean = {np.mean(norm_residuals):.3f}\n"
        f"std = {np.std(norm_residuals):.3f}",
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

NameError: name 'result' is not defined

---

## 4. Physical Quantities: Einstein Mass and Radius <a id="4-physical-quantities"></a>

### Extracting Science from the Model

The Einstein radius and enclosed mass are the primary science outputs.
For an SIE, the mass within $\theta_E$ is (C&K eq. 4.26):

$$
M(\theta_E) = \pi \theta_E^2 \Sigma_{\rm cr} D_d^2
$$

In [7]:
# ============================================================
# PHYSICAL QUANTITIES
# ============================================================
ml = result.max_log_likelihood_instance
cosmo = al.cosmo.Planck15()

theta_E = ml.galaxies.lens.mass.einstein_radius
z_d = 0.5
z_s = 1.0

# Physical Einstein radius in kpc
kpc_per_arcsec = cosmo.kpc_per_arcsec_from(redshift=z_d)
R_E_kpc = theta_E * kpc_per_arcsec

# Critical surface density
Sigma_cr = cosmo.critical_surface_density_between_redshifts_solar_mass_per_kpc2_from(
    redshift_0=z_d, redshift_1=z_s
)

# Enclosed mass
M_E = np.pi * R_E_kpc**2 * Sigma_cr

print(f"Einstein radius: θ_E = {theta_E:.4f} arcsec = {R_E_kpc:.2f} kpc")
print(f"Critical surface density: Σ_cr = {Sigma_cr:.3e} M_sun/kpc²")
print(f"Enclosed mass: M(<θ_E) = {M_E:.3e} M_sun")
print(f"\nShear: (γ₁, γ₂) = ({ml.galaxies.lens.shear.gamma_1:.4f}, "
      f"{ml.galaxies.lens.shear.gamma_2:.4f})")
print(f"|γ| = {np.sqrt(ml.galaxies.lens.shear.gamma_1**2 + ml.galaxies.lens.shear.gamma_2**2):.4f}")

NameError: name 'result' is not defined

---

## 5. Source Plane Reconstruction <a id="5-source-plane"></a>

In [8]:
# ============================================================
# SOURCE-PLANE AND TRACER SUBPLOT
# ============================================================
tracer_plotter = aplt.TracerPlotter(
    tracer=result.max_log_likelihood_tracer,
    grid=result.grids.lp,
)
tracer_plotter.subplot_tracer()

NameError: name 'result' is not defined

---

## 6. Publication-Quality Figures <a id="6-publication-figures"></a>

### Customizing PyAutoLens Plots

PyAutoLens plots wrap matplotlib, so you can customize everything via
`MatPlot2D` and `Visuals2D`.

In [9]:
# ============================================================
# PUBLICATION-QUALITY FIGURE
# ============================================================
# Customize colors, labels, and layout for a journal figure.
# TODO: Adjust for your specific paper's style guide.
# ============================================================

mat_plot = aplt.MatPlot2D(
    title=aplt.Title(label="Best-Fit Lens Model", fontsize=14),
    ylabel=aplt.YLabel(label="y [arcsec]"),
    xlabel=aplt.XLabel(label="x [arcsec]"),
    cmap=aplt.Cmap(cmap="inferno"),
    colorbar=aplt.Colorbar(label="Surface Brightness"),
    # output=aplt.Output(path="figures/", format="pdf"),  # Uncomment to save
)

tracer_plotter = aplt.TracerPlotter(
    tracer=result.max_log_likelihood_tracer,
    grid=result.grids.lp,
    mat_plot_2d=mat_plot,
)
tracer_plotter.figures_2d(image=True)

NameError: name 'result' is not defined

In [10]:
# ============================================================
# MULTI-PANEL FIGURE (data | model | residuals)
# ============================================================
# The classic 3-panel figure for lens modeling papers.
# ============================================================

fit = result.max_log_likelihood_fit

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

extent = [-3, 3, -3, 3]  # arcsec

# Panel 1: Data
im0 = axes[0].imshow(fit.data.native, origin="lower", cmap="inferno", extent=extent)
axes[0].set_title("Data", fontsize=13)
axes[0].set_xlabel("x [arcsec]")
axes[0].set_ylabel("y [arcsec]")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Panel 2: Model
im1 = axes[1].imshow(fit.model_data.native, origin="lower", cmap="inferno", extent=extent)
axes[1].set_title("Model", fontsize=13)
axes[1].set_xlabel("x [arcsec]")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Panel 3: Normalized residuals
im2 = axes[2].imshow(fit.normalized_residual_map.native, origin="lower",
                      cmap="RdBu_r", vmin=-3, vmax=3, extent=extent)
axes[2].set_title("Normalized Residuals", fontsize=13)
axes[2].set_xlabel("x [arcsec]")
plt.colorbar(im2, ax=axes[2], shrink=0.8, label=r"$(d-m)/\sigma$")

plt.suptitle(f"$\\theta_E = {theta_E:.3f}''$, "
             f"$\\chi^2_{{red}} = {chi2_red:.3f}$",
             fontsize=14, y=1.02)
plt.tight_layout()
# plt.savefig("figures/data_model_residuals.pdf", bbox_inches='tight')  # Uncomment to save
plt.show()

NameError: name 'result' is not defined

---

## 7. Model Comparison: Bayesian Evidence <a id="7-model-comparison"></a>

### Using the Evidence for Model Selection

The Bayesian evidence $\mathcal{Z}$ penalizes unnecessary complexity.
For two models $M_1$ and $M_2$, the **Bayes factor** is:

$$
\ln B_{12} = \ln \mathcal{Z}_1 - \ln \mathcal{Z}_2
$$

| $\ln B_{12}$ | Interpretation |
|-------------|----------------|
| < 1 | Not significant |
| 1–2.5 | Moderate evidence for $M_1$ |
| 2.5–5 | Strong evidence |
| > 5 | Decisive evidence |

In [11]:
# ============================================================
# BAYESIAN EVIDENCE
# ============================================================
# TODO: Compare evidence between different mass models
# (e.g., SIE vs PowerLaw vs composite) to determine which
# model the data supports.
# ============================================================

log_evidence = result.samples.log_evidence
print(f"Log evidence: ln(Z) = {log_evidence:.2f}")
print(f"\nTo compare models, run two fits and compute:")
print(f"  ln B = ln Z_1 - ln Z_2")
print(f"  |ln B| > 5 → decisive preference for one model")

NameError: name 'result' is not defined

---

## 8. Exporting Results to CSV and FITS <a id="8-exporting"></a>

In [12]:
# ============================================================
# EXPORT RESULTS
# ============================================================
# Save key parameters and uncertainties for downstream analysis.
# ============================================================

import json

# Build a results dictionary
results_dict = {
    "target": "example_lens",
    "z_lens": z_d,
    "z_source": z_s,
    "theta_E_arcsec": float(theta_E),
    "R_E_kpc": float(R_E_kpc),
    "M_E_solar": float(M_E),
    "gamma_1": float(ml.galaxies.lens.shear.gamma_1),
    "gamma_2": float(ml.galaxies.lens.shear.gamma_2),
    "chi_squared_red": float(chi2_red),
    "log_evidence": float(log_evidence),
}

# Save to JSON
output_path = Path("output") / "module_08"
output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / "results_summary.json", "w") as f:
    json.dump(results_dict, f, indent=2)

print("Results saved to output/module_08/results_summary.json")
print(json.dumps(results_dict, indent=2))

NameError: name 'z_d' is not defined

---

## 9. Exercises <a id="9-exercises"></a>

### Exercise 1: Parameter Uncertainties

Extract the 16th, 50th, and 84th percentile values for $\theta_E$ from
`result.samples`. Report $\theta_E = \text{median}^{+\sigma_+}_{-\sigma_-}$.

### Exercise 2: Model Comparison

Fit the same dataset with (a) SIS, (b) SIE, (c) SIE + shear. Compute the
Bayes factors. Does the data require ellipticity? External shear?

### Exercise 3: Multi-Panel Publication Figure

Create a 6-panel figure showing: (1) data, (2) lens light model, (3) lens-subtracted
data, (4) source model, (5) source-plane reconstruction, (6) convergence map with
critical curves overlaid. Save as PDF at 300 dpi.

### Exercise 4: Parameter Table

Create a LaTeX-formatted parameter table (using `astropy.table` or pandas)
with columns: parameter name, maximum likelihood value, median, ±1σ.
This is the standard table for lens modeling papers.

---

## Summary

| Task | Tool | Output |
|------|------|--------|
| Parameter summary | `result.info` | Text summary |
| Corner plots | `aplt.NestPlotter.corner_anesthetic()` | Posterior visualization |
| Residuals | `aplt.FitImagingPlotter.subplot_fit()` | Diagnostic figure |
| Einstein mass | Cosmology + $\theta_E$ | Physical mass |
| Publication figures | `aplt.MatPlot2D` + matplotlib | Journal-ready plots |
| Model comparison | `result.samples.log_evidence` | Bayes factor |
| Export | JSON / CSV / FITS | Reproducible results |

---

## Congratulations!

You've completed all 8 modules of **Learning to Autolens**. You now know how to:
1. Set up grids, galaxies, and ray-tracing (Module 01)
2. Simulate realistic lens data (Module 02)
3. Fit a Bayesian lens model (Module 03)
4. Use the SLaM pipeline for robust modeling (Module 04)
5. Reconstruct sources non-parametrically (Module 05)
6. Decompose mass into stars + dark matter (Module 06)
7. Prepare and model real data (Module 07)
8. Extract and publish your results (Module 08)

**Where to go next:**
- Apply SLaM to your own AGEL targets
- Explore subhalo detection (autolens_workspace/slam/subhalo/)
- Combine lensing with kinematics (velocity dispersion from ppxf)
- Multi-wavelength modeling (autolens_workspace/advanced/multi/)

---

*Learning to Autolens — Module 08*
*Rodrigo Córdova Rosado, Harvard CfA*
*Built with Claude Code*

---

# Solutions

*Complete worked solutions with commentary for all exercises in this module.*

### Solution 1: Parameter Uncertainties

The standard reporting convention in lens modeling papers is the **median and 68% credible interval** from the posterior:
$$\theta_E = \text{median}^{+\sigma_+}_{-\sigma_-}$$
where $\sigma_+ = P_{84} - P_{50}$ and $\sigma_- = P_{50} - P_{16}$.

For a Gaussian posterior, $\sigma_+ \approx \sigma_- \approx 1\sigma$. Asymmetric error bars indicate a skewed posterior (common for parameters like intensity that have a hard lower bound).

In [13]:
# ============================================================
# SOLUTION 1: PARAMETER UNCERTAINTIES (template)
# ============================================================
# After running the fit:
# import numpy as np
# theta_E_samples = result.samples.values[:, idx_theta_E]
# p16, p50, p84 = np.percentile(theta_E_samples, [16, 50, 84])
# print(f"theta_E = {p50:.4f} +{p84-p50:.4f} -{p50-p16:.4f} arcsec")
print("Report: median +sigma_plus -sigma_minus from posterior samples.")

Report: median +sigma_plus -sigma_minus from posterior samples.


### Solution 2: Model Comparison

The Bayes factor $\ln B = \ln \mathcal{Z}_1 - \ln \mathcal{Z}_2$ is the gold standard for model comparison in lens modeling. Key interpretation:
- $|\ln B| < 1$: inconclusive (data doesn't distinguish the models)
- $|\ln B| \sim 2.5$: moderate preference
- $|\ln B| > 5$: decisive

**Example questions to answer with model comparison:**
- SIS vs SIE: is the lens elliptical?
- SIE vs PowerLaw: is the density slope $\neq 2$?
- Single source vs double source: is there a second source plane?

### Solution 3: Multi-Panel Publication Figure

The standard lens modeling figure has 3-6 panels showing data, model, residuals, source reconstruction, and convergence. Key formatting tips:
- Use consistent color scales across panels
- Label all axes with units (arcsec)
- Include $\theta_E$ and $\chi^2_{\rm red}$ in the caption
- Save as vector format (PDF) for publication

In [14]:
# ============================================================
# SOLUTION 3: PUBLICATION FIGURE (template)
# ============================================================
# After running the fit:
# fit = result.max_log_likelihood_fit
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# extent = [-3, 3, -3, 3]
#
# axes[0].imshow(fit.data.native, origin="lower", cmap="inferno", extent=extent)
# axes[0].set_title("Data")
# axes[1].imshow(fit.model_data.native, origin="lower", cmap="inferno", extent=extent)
# axes[1].set_title("Model")
# axes[2].imshow(fit.normalized_residual_map.native, origin="lower",
#                cmap="RdBu_r", vmin=-3, vmax=3, extent=extent)
# axes[2].set_title("Residuals")
# for ax in axes:
#     ax.set_xlabel("x [arcsec]"); ax.set_ylabel("y [arcsec]")
# plt.savefig("data_model_residuals.pdf", bbox_inches='tight', dpi=300)
print("Save as PDF at 300 dpi for publication quality.")

Save as PDF at 300 dpi for publication quality.


### Solution 4: LaTeX Parameter Table

A properly formatted parameter table is essential for reproducibility. Include: parameter name (with LaTeX symbol), maximum likelihood value, median, and asymmetric error bars. Use `astropy.table` or pandas for automated generation.

In [15]:
# ============================================================
# SOLUTION 4: LATEX TABLE (template)
# ============================================================
# After running the fit:
# print(r"\begin{tabular}{lcccc}")
# print(r"Parameter & ML value & Median & $-1\sigma$ & $+1\sigma$ \\")
# print(r"\hline")
# for name, idx in param_indices.items():
#     vals = result.samples.values[:, idx]
#     ml_val = result.max_log_likelihood_instance...
#     p16, p50, p84 = np.percentile(vals, [16, 50, 84])
#     print(f"{name} & {ml_val:.4f} & {p50:.4f} & {p50-p16:.4f} & {p84-p50:.4f} \\\\")
# print(r"\end{tabular}")
print("Automate table generation for reproducibility across targets.")

Automate table generation for reproducibility across targets.
